# 03 · JTAGulator-style SWD pinout detection

Just like a JTAGulator, FaultyCat can **brute-force a target's debug pinout** for you. It walks every ordered pair of the 8 scanner-header channels (GP0..GP7) — that's P(8,2)=56 permutations — and drives an SWD handshake on each one until a valid DPIDR comes back. When it does, you know exactly which pins are **SWCLK** and **SWDIO**, no datasheet required.

Everything here runs through `cat.scanner` over the scanner-shell CDC, and there's no high voltage anywhere in sight.

## Wiring

Hook the target's candidate debug pads up to the **scanner header**:

- Target **SWCLK** → any GP channel (GP0..GP7)
- Target **SWDIO** → any other GP channel
- A common **GND** between the target and FaultyCat

Don't worry about which pad is which — figuring that out is the whole point of the scan.

In [ ]:
import faultycat as fc
SIM = False                 # True = simulator (returns a canned GP2/GP3 match)
cat = fc.connect(simulator=SIM)
cat.scanner                 # shows scanner capabilities

## 1 · Scan for SWD

`swd()` sweeps all 56 permutations, and `on_progress` streams the firmware's progress lines as it goes. If you get `NO_MATCH`, nothing valid turned up — double-check your wiring, make sure the target is powered, and confirm the pins actually land within GP0..GP7.

In [ ]:
res = cat.scanner.swd(timeout_s=45, on_progress=print)
res                          # SwdScanResult: matched / swclk_gp / swdio_gp

In [ ]:
if res.matched:
    print(f'Found SWD:  SWCLK = GP{res.swclk_gp}   SWDIO = GP{res.swdio_gp}')
    print('Raw firmware lines (DPIDR / targetsel):')
    for line in res.lines:
        print('   ', line)
else:
    print('No SWD match. Nothing wired to the scanner header, wrong pins, or target unpowered.')

## 2 · (Bonus) I2C bus discovery

The same header can brute-force an I2C bus too — it hunts down SDA/SCL and lists the device addresses that ACK.

In [ ]:
try:
    i2c = cat.scanner.i2c(timeout_s=30)
    if i2c.matched:
        print(f'I2C on SDA=GP{i2c.sda_gp} SCL=GP{i2c.scl_gp} — addresses: {i2c.addresses_hex}')
    else:
        print('No I2C devices found.')
except NotImplementedError as e:
    print('I2C scan needs the full faultycmd:', e)

In [ ]:
cat.close()